**Shazeer et al., 2017, _Outrageously Large Neural Networks: The Sparsely-Gated Mixture-of-Experts Layer_** 

为了避免一个容易混淆的地方，先给全文最重要的定位：

$$
\boxed{
\text{2017 Sparse MoE 的核心目标不是重新建立概率混合模型，}
}
$$

$$
\boxed{
\text{而是实现 Conditional Computation：总参数很多，但每次只计算少数参数。}
}
$$

---

# 一、论文要解决什么问题？

## 1. Dense 网络的 Scaling 问题

传统神经网络增加容量，最直接的方法就是增加：

$$
\text{层数、hidden dim、FFN dim}
$$

但这通常导致：

$$
\text{Parameters}\uparrow
\quad\Rightarrow\quad
\text{FLOPs}\uparrow
$$

例如一个 FFN：

$$
h\in\mathbb R^d
$$

经过：

$$
FFN(h)
=
W_2\phi(W_1h)
$$

如果：

$$
W_1\in\mathbb R^{d_{ff}\times d}
$$

那么参数量和计算量都大约正比于：

$$
d\,d_{ff}
$$

所以 Dense Model 基本上是：

$$
\boxed{
\text{模型容量增加}
\Longrightarrow
\text{每个样本计算量增加}
}
$$

---

# 二、作者的核心思想：Conditional Computation

作者提出一个很自然的问题：

> 为什么模型拥有的所有参数，每次 forward 都必须使用？

假设我们有：

$$
N=1000
$$

个 Expert。

但是对于一个输入 \(x\)，只选择：

$$
K=4
$$

个 Expert。

那么：

$$
\boxed{
\text{Total Parameters}
\sim N
}
$$

但：

$$
\boxed{
\text{Active Computation}
\sim K
}
$$

其中：

$$
K\ll N
$$

于是理论上可以：

$$
N:100\rightarrow1000\rightarrow10000
$$

不断增加模型容量，而每个输入仍然只计算少数 Expert。

这就是：

$$
\boxed{\text{Conditional Computation}}
$$

也是这篇论文最重要的出发点。

---

# 三、MoE Layer 的基本结构

定义：

$$
N
$$

个 Expert：

$$
E_1(x),E_2(x),\cdots,E_N(x)
$$

以及一个 Gate：

$$
G(x)
=
[G_1(x),\cdots,G_N(x)]
$$

MoE 输出：

$$
\boxed{
y
=
\sum_{i=1}^{N}
G_i(x)E_i(x)
}
$$

如果普通 Softmax：

$$
G(x)=Softmax(xW_g)
$$

那么：

$$
G_i(x)>0
$$

几乎对所有 \(i\) 成立。

于是仍然必须计算：

$$
E_1(x),E_2(x),\cdots,E_N(x)
$$

这没有真正降低计算量。

所以关键要求是：

$$
\boxed{G(x)\text{ 必须是 sparse}}
$$

例如：

$$
G(x)
=
[0,\ 0,\ 0.6,\ 0,\ 0.4,\ 0]
$$

那么：

$$
y
=
0.6E_3(x)
+
0.4E_5(x)
$$

只需要计算：

$$
E_3,E_5
$$

这就是 **Sparsely-Gated MoE**。

---

# 四、Noisy Top-K Gating

这是论文第一个核心技术。

作者定义：

$$
\boxed{
G(x)
=
Softmax(KeepTopK(H(x),K))
}
$$

其中：

$$
\boxed{
H_i(x)
=
(xW_g)_i
+
\epsilon_i
Softplus((xW_{noise})_i)
}
$$

并且：

$$
\epsilon_i\sim\mathcal N(0,1)
$$

我们一步一步拆。

---

# 五、第一步：Clean Router Logit

定义：

$$
\mu_i(x)
=
(xW_g)_i
$$

所以：

$$
\mu(x)=xW_g
$$

例如：

$$
\mu(x)
=
[1.2,0.7,2.0,1.5]
$$

表示 Router 对四个 Expert 的原始 routing score。

---

# 六、第二步：学习 Noise Scale

另外一个矩阵：

$$
W_{noise}
$$

计算：

$$
s_i=(xW_{noise})_i
$$

再经过：

$$
\sigma_i
=
Softplus(s_i)
$$

其中：

$$
Softplus(z)=\log(1+e^z)
$$

这样：

$$
\sigma_i>0
$$

因此每个 Expert 不仅有一个：

$$
\mu_i
$$

还有：

$$
\sigma_i
$$

---

# 七、第三步：Gaussian Noise

采样：

$$
\epsilon_i\sim N(0,1)
$$

得到：

$$
\boxed{
H_i
=
\mu_i+\sigma_i\epsilon_i
}
$$

所以：

$$
\boxed{
H_i
\sim
N(\mu_i,\sigma_i^2)
}
$$

注意我们前面特别讨论过：

这里服从 Gaussian 的是：

$$
\boxed{\text{Noisy Router Logit }H_i}
$$

不是最终 Softmax probability。

---

# 八、第四步：Top-K

假设：

$$
H=
[1.2,0.7,2.0,1.5]
$$

并且：

$$
K=2
$$

Top-2 是：

$$
2.0,\quad1.5
$$

因此：

$$
KeepTopK(H,2)
=
[-\infty,-\infty,2.0,1.5]
$$

---

# 九、第五步：Sparse Softmax

再：

$$
G(x)
=
Softmax(
[-\infty,-\infty,2.0,1.5]
)
$$

得到大约：

$$
G(x)
=
[0,0,0.622,0.378]
$$

因此：

$$
\boxed{
y
=
0.622E_3(x)
+
0.378E_4(x)
}
$$

只执行：

$$
E_3,E_4
$$

---

# 十、一次完整 forward

所以完整 forward 是：

$$
x
$$

↓

$$
\mu=xW_g
$$

↓

$$
\sigma=Softplus(xW_{noise})
$$

↓

$$
H=\mu+\sigma\odot\epsilon
$$

↓

$$
S=TopK(H,K)
$$

↓

$$
G=Softmax(KeepTopK(H,K))
$$

↓

只计算：

$$
E_i(x),\qquad i\in S
$$

↓

最终：

$$
\boxed{
y
=
\sum_{i\in S}
G_iE_i(x)
}
$$

这是整篇论文最核心的一条计算链。

---

# 十一、Expert 是什么？

论文里的 Expert 本身并不复杂。

基本就是：

$$
\boxed{\text{Feed-Forward Network}}
$$

可以理解为：

$$
E_i(x)
=
W_{2,i}\phi(W_{1,i}x+b_{1,i})+b_{2,i}
$$

Expert：

$$
E_1,\dots,E_N
$$

网络结构相同，但：

$$
\boxed{
\theta_1,\dots,\theta_N
}
$$

参数独立。

所以模型增加 Expert：

$$
N\uparrow
$$

总参数量就可以大幅增加。

---

# 十二、这里已经和 1991 MoE 有本质区别

1991 MoE 是概率模型：

$$
p(y|x)
=
\sum_i
p_i(x)p(y|x,z=i)
$$

然后：

$$
L
=
-\log
\sum_i p_iq_i
$$

并得到 posterior responsibility：

$$
r_i
=
\frac{p_iq_i}{\sum_jp_jq_j}
$$

---

2017 Sparse MoE 已经把 MoE 当成普通神经网络 Layer：

$$
\boxed{
y=\sum_iG_iE_i(x)
}
$$

然后：

$$
y
\rightarrow
\text{后续网络}
\rightarrow
L_{task}
$$

所以：

$$
\boxed{\text{不需要 Gaussian mixture responsibility}}
$$

而是：

$$
\boxed{\text{最终任务Loss直接反向传播}}
$$

---

# 十三、主任务 Loss 怎么训练 Expert？

这是第一个重要推导。

设 Top-K 集合：

$$
S
$$

MoE 输出：

$$
y
=
\sum_{i\in S}
g_ie_i
$$

其中：

$$
e_i=E_i(x)
$$

假设后面的网络最终产生：

$$
L_{task}
$$

先得到：

$$
\delta
=
\frac{\partial L}{\partial y}
$$

那么：

$$
\frac{\partial y}{\partial e_i}
=
g_i
$$

所以：

$$
\boxed{
\frac{\partial L}{\partial e_i}
=
g_i\delta
}
$$

再：

$$
e_i=E_i(x;\theta_i)
$$

因此：

$$
\boxed{
\frac{\partial L}{\partial\theta_i}
=
g_i
\delta
\frac{\partial E_i(x)}
{\partial\theta_i}
}
$$

所以：

$$
\boxed{
\text{Gate权重越大的Expert，一般获得越强的该样本训练信号}
}
$$

而：

$$
i\notin S
$$

则：

$$
g_i=0
$$

当前样本不会通过主任务路径训练该 Expert。

---

# 十四、Router 怎么得到主任务梯度？

这是第二个重要推导。

仍然：

$$
y
=
\sum_{i\in S}g_ie_i
$$

所以：

$$
\frac{\partial L}{\partial g_i}
=
\left\langle
\delta,e_i
\right\rangle
$$

其中：

$$
\delta=\frac{\partial L}{\partial y}
$$

现在对于选中的 Expert：

$$
g_i
=
\frac{e^{H_i}}
{\sum_{j\in S}e^{H_j}}
$$

Softmax derivative：

$$
\frac{\partial g_j}{\partial H_i}
=
g_j(\delta_{ij}-g_i)
$$

因此：

$$
\frac{\partial L}{\partial H_i}
=
\sum_{j\in S}
\frac{\partial L}{\partial g_j}
\frac{\partial g_j}{\partial H_i}
$$

代入：

$$
=
\sum_j
\langle\delta,e_j\rangle
g_j(\delta_{ij}-g_i)
$$

整理：

$$
=
g_i\langle\delta,e_i\rangle
-
g_i
\sum_jg_j
\langle\delta,e_j\rangle
$$

因为：

$$
\sum_jg_je_j=y
$$

所以：

$$
\boxed{
\frac{\partial L}{\partial H_i}
=
g_i
\left\langle
\frac{\partial L}{\partial y},
e_i-y
\right\rangle
}
$$

这就是 Router 从任务 Loss 获得的信号。

直观上：

> 如果提高 Expert \(i\) 的 routing weight 能降低最终任务 Loss，那么梯度就会提高它的 routing score。

---

# 十五、Top-K 本身怎么办？

关键问题来了：

$$
S=TopK(H,K)
$$

本身是：

$$
\boxed{\text{离散选择}}
$$

所以不能简单地对：

$$
\mathbf1[i\in TopK]
$$

做普通连续求导。

但是一旦当前：

$$
S
$$

确定，在 Top-K 集合不发生变化的局部区域：

$$
G_S=Softmax(H_S)
$$

仍然可导。

所以主任务 Loss 可以：

$$
L_{task}
\rightarrow
G_i
\rightarrow
H_i
\rightarrow
W_g
$$

训练当前选中的 routing weights。

但这还不够。

因为会出现：

$$
\boxed{\text{Load Imbalance}}
$$

---

# 十六、为什么会 Expert Collapse？

假设 Expert 1 初期碰巧好一点。

于是：

$$
E_1
$$

更容易进入 Top-K。

↓

它获得更多训练样本。

↓

$$
E_1
$$

训练得更好。

↓

Router 更喜欢它。

↓

它获得更多样本。

形成：

$$
\boxed{
\text{被选择}
\rightarrow
\text{训练更多}
\rightarrow
\text{更强}
\rightarrow
\text{更容易被选择}
}
$$

最终可能：

$$
N=1000
$$

但大量 token 都只集中到少数 Expert。

因此作者必须额外做：

$$
\boxed{\text{Load Balancing}}
$$

---

# 十七、Importance Loss

首先定义一个 batch：

$$
X=\{x_1,\dots,x_B\}
$$

注意：

$$
\boxed{X=\text{batch里的输入集合}}
$$

不是 Gate 输出。

对于每个输入：

$$
G(x_j)
=
[G_1(x_j),\dots,G_N(x_j)]
$$

定义 Expert \(i\) 的 Importance：

$$
\boxed{
Importance_i(X)
=
\sum_{x\in X}G_i(x)
}
$$

例如：

$$
G(x_1)=[0.7,0.3,0,0]
$$

$$
G(x_2)=[0.6,0,0.4,0]
$$

$$
G(x_3)=[0,0.8,0.2,0]
$$

$$
G(x_4)=[0,0,0.3,0.7]
$$

那么：

$$
Importance
=
[1.3,1.1,0.9,0.7]
$$

---

# 十八、如何衡量 Importance 是否均衡？

论文使用 coefficient of variation：

$$
CV(v)
=
\frac{std(v)}{mean(v)}
$$

所以：

$$
\boxed{
L_{importance}
=
w_{importance}
CV(Importance(X))^2
}
$$

即：

$$
L_{importance}
=
w_{importance}
\frac{Var(Importance)}
{Mean(Importance)^2}
$$

如果：

$$
Importance_1=\cdots=Importance_N
$$

那么：

$$
Var=0
$$

因此：

$$
L_{importance}=0
$$

注意它要求的是：

$$
\boxed{\text{整个batch上所有Expert之间总体均衡}}
$$

不是要求某一个 token 的 Top-K 权重均匀。

一个 token 完全可以：

$$
[0,0,0.9,0,0.1]
$$

没问题。

---

# 十九、为什么 Importance Loss 还不够？

因为：

$$
\boxed{
Importance\neq实际计算负载
}
$$

例如 Expert 1：

100 个 token，每个：

$$
g_1=0.01
$$

所以：

$$
Importance_1=1
$$

Expert 2：

1 个 token：

$$
g_2=1
$$

所以：

$$
Importance_2=1
$$

于是：

$$
Importance=[1,1]
$$

看起来完美均衡。

但是：

$$
Load_1=100
$$

$$
Load_2=1
$$

实际计算量严重失衡。

因此需要：

$$
\boxed{Load Loss}
$$

---

# 二十、最自然的 Hard Load

最直观：

$$
\boxed{
Load_i^{hard}(X)
=
\sum_{x\in X}
\mathbf1[i\in TopK(x)]
}
$$

它就是：

> Expert \(i\) 实际收到多少个 token。

这个定义本身没有问题。

问题是：

$$
\boxed{\mathbf1[i\in TopK]\text{不可导}}
$$

---

# 二十一、为什么 Hard Load 不可导？

假设 Top-1：

$$
H_1=1.01,\qquad H_2=1.00
$$

Expert 1 被选：

$$
Load_1=1
$$

现在：

$$
H_1:1.01\rightarrow1.001
$$

还是：

$$
Load_1=1
$$

甚至：

$$
H_1=1.000001
$$

还是：

$$
1
$$

直到：

$$
H_1<1
$$

突然：

$$
Load_1:1\rightarrow0
$$

因此：

$$
\boxed{
\text{Hard Load 是阶跃函数}
}
$$

梯度几乎处处为 0。

所以不能很好地训练 Router。

---

# 二十二、为什么前面要加 Gaussian Noise？

现在 Noisy Gate 的另一个作用出现了。

因为：

$$
H_i
=
\mu_i+\sigma_i\epsilon_i
$$

所以：

$$
H_i
\sim
N(\mu_i,\sigma_i^2)
$$

那么：

$$
i\in TopK
$$

虽然是离散事件，但：

$$
\boxed{
P(i\in TopK)
}
$$

可以变成连续概率。

---

# 二十三、Top-1 的 threshold 推导

先看最简单的 Top-1。

对于 Expert \(i\)，其他 Expert 的最高 score：

$$
T_i
=
\max_{j\neq i}H_j
$$

那么：

$$
i\in Top1
$$

等价于：

$$
H_i>T_i
$$

由于：

$$
H_i=\mu_i+\sigma_i\epsilon_i
$$

所以：

$$
\mu_i+\sigma_i\epsilon_i>T_i
$$

整理：

$$
\epsilon_i>
\frac{T_i-\mu_i}{\sigma_i}
$$

因为：

$$
\epsilon_i\sim N(0,1)
$$

所以：

$$
P(i\in Top1)
=
1-
\Phi
\left(
\frac{T_i-\mu_i}{\sigma_i}
\right)
$$

利用：

$$
1-\Phi(a)=\Phi(-a)
$$

得到：

$$
\boxed{
P(i\in Top1)
=
\Phi
\left(
\frac{\mu_i-T_i}{\sigma_i}
\right)
}
$$

这里：

$$
\Phi
$$

是标准正态 CDF。

---

# 二十四、这里是真正的概率

注意：

$$
\mathcal N(h;\mu,\sigma^2)
$$

是：

$$
\text{概率密度}
$$

而：

$$
P(H_i>T_i)
=
\int_{T_i}^{\infty}
\mathcal N(h;\mu_i,\sigma_i^2)dh
$$

是：

$$
\boxed{\text{真正的概率}}
$$

所以：

$$
0\le P(i\in TopK)\le1
$$

这正是我们前面讨论的重点。

---

# 二十五、Top-K 的 threshold

如果：

$$
K>1
$$

思想完全一样。

Expert \(i\) 想进入 Top-K，就必须超过一个排名边界：

$$
\boxed{
T_i
=
\text{其他Experts中进入Top-K所对应的竞争阈值}
}
$$

简化理解就是：

$$
\text{其他Expert score中的第K大值}
$$

因此仍然：

$$
i\in TopK
\iff
H_i>T_i
$$

于是：

$$
\boxed{
P(i\in TopK)
=
\Phi
\left(
\frac{\mu_i-T_i}{\sigma_i}
\right)
}
$$

实际论文计算时，会根据当前 Expert 是否已经在 Top-K 中使用相应的 \(k\) / \(k+1\) 排名阈值，以构造条件入选概率。

---

# 二十六、Smooth Load Estimator

现在就非常自然了。

原来：

$$
Load_i^{hard}
=
\sum_x
\mathbf1[i\in TopK]
$$

改成：

$$
\boxed{
Load_i(X)
=
\sum_{x\in X}
P(i\in TopK|x)
}
$$

于是：

$$
Load_i
$$

表示：

$$
\boxed{\text{Expert }i\text{预计会收到多少个token}}
$$

例如：

$$
Load
=
[24.2,25.3,24.8,25.7]
$$

而不是硬计数：

$$
[24,25,25,26]
$$

前者对 Router 参数是平滑得多的。

---

# 二十七、Load Loss

和 Importance 一样：

$$
\boxed{
L_{load}
=
w_{load}
CV(Load(X))^2
}
$$

即：

$$
L_{load}
=
w_{load}
\frac{Var(Load)}
{Mean(Load)^2}
$$

希望：

$$
Load_1\approx\cdots\approx Load_N
$$

---

# 二十八、Importance 和 Load 到底有什么区别？

这个一定要记牢：

$$
\boxed{
Importance_i
=
\sum_xG_i(x)
}
$$

衡量：

$$
\boxed{\text{Expert拿到了多少Gate权重}}
$$

而：

$$
\boxed{
Load_i
=
\sum_xP(i\in TopK|x)
}
$$

衡量：

$$
\boxed{\text{Expert预计要处理多少token}}
$$

所以：

$$
L_{importance}
$$

主要平衡：

$$
\text{routing mass}
$$

而：

$$
L_{load}
$$

主要平衡：

$$
\text{computation load}
$$

---

# 二十九、最终 Loss

现在终于可以把训练目标完整写出来：

$$
\boxed{
L_{total}
=
L_{task}
+
L_{importance}
+
L_{load}
}
$$

展开：

$$
\boxed{
L_{total}
=
L_{task}
+
w_{importance}
CV(Importance(X))^2
+
w_{load}
CV(Load(X))^2
}
$$

其中：

### \(L_{task}\)

负责：

$$
\boxed{\text{把最终任务做好}}
$$

训练：

$$
Experts+Router
$$

### \(L_{importance}\)

负责：

$$
\boxed{\text{Gate权重不要集中到少数Expert}}
$$

### \(L_{load}\)

负责：

$$
\boxed{\text{Token计算负载不要集中到少数Expert}}
$$

---

# 三十、一次训练完整发生了什么？

一个 batch：

$$
X
$$

进入 Router：

$$
\mu=XW_g
$$

$$
\sigma=Softplus(XW_{noise})
$$

采样：

$$
\epsilon\sim N(0,I)
$$

得到：

$$
H=\mu+\sigma\odot\epsilon
$$

然后：

$$
TopK(H)
$$

得到 sparse Gate：

$$
G
$$

根据 \(G\)：

$$
\boxed{\text{dispatch tokens}}
$$

到对应 Expert。

每个 Expert：

$$
E_i(X_i)
$$

计算自己的 token。

然后：

$$
\boxed{\text{combine}}
$$

得到：

$$
Y
$$

送入后续网络。

最终产生：

$$
L_{task}
$$

与此同时 batch 内统计：

$$
Importance
$$

以及 smooth：

$$
Load
$$

得到：

$$
L_{importance},L_{load}
$$

最后：

$$
\boxed{
L_{total}.backward()
}
$$

一次完成 Router + Expert 的训练。

---

# 三十一、为什么 Noise 非常关键？

现在回头看：

$$
H_i
=
\mu_i+\sigma_i\epsilon_i
$$

Noise 实际承担两个作用。

第一：

$$
\boxed{\text{Exploration}}
$$

如果两个 Expert：

$$
\mu_1=1.01,\quad\mu_2=1.00
$$

没有 noise：

$$
E_1
$$

可能一直胜出。

加 noise 后：

$$
E_2
$$

也有机会进入 Top-K，得到训练。

第二：

$$
\boxed{\text{Smooth Load Estimation}}
$$

因为：

$$
H_i\sim N(\mu_i,\sigma_i^2)
$$

所以离散事件：

$$
i\in TopK
$$

才能转成：

$$
P(i\in TopK)
$$

并通过 Gaussian CDF 得到平滑 estimator。

所以 Noisy Top-K 不是简单“为了随机”。

---

# 三十二、它和重参数化有什么关系？

形式：

$$
H_i
=
\mu_i+\sigma_i\epsilon_i
$$

确实与 VAE 的：

$$
z=\mu+\sigma\epsilon
$$

非常像。

所以可以认为：

$$
\boxed{\text{使用了重参数化形式的Gaussian noisy score}}
$$

但是不要说：

> Top-K 被重参数化成了 Gaussian。

更准确的是：

$$
\boxed{
\text{Router score被Gaussian随机化}
}
$$

然后：

$$
\boxed{
\text{Top-K离散事件被写成跨越threshold的概率}
}
$$

即：

$$
\mathbf1[H_i>T_i]
$$

转成：

$$
P(H_i>T_i)
$$

---

# 三十三、Shrinking Batch Problem

理论上已经很好了，但工程上出现一个问题。

假设：

$$
B=1024
$$

$$
N=1024
$$

$$
K=4
$$

总 assignments：

$$
BK=4096
$$

平均每个 Expert：

$$
\frac{BK}{N}=4
$$

个 token。

于是 Expert FFN 可能只能做：

$$
[4,d]\times[d,d_{ff}]
$$

这种非常小的矩阵乘法。

GPU/TPU 利用率会很差。

这就是：

$$
\boxed{\text{Shrinking Batch Problem}}
$$

---

# 三十四、解决思路：把同一 Expert 的 token 聚起来

假设：

$$
x_1,x_7,x_{23},x_{50}
$$

都去了：

$$
E_3
$$

不要四次：

$$
E_3(x_1)
$$

$$
E_3(x_7)
$$

……

而是：

$$
\boxed{
E_3(
[x_1,x_7,x_{23},x_{50}]
)
}
$$

一次 batch computation。

所以整个 MoE 工程结构逐渐形成：

$$
\boxed{
Dispatch
\rightarrow
Expert Compute
\rightarrow
Combine
}
$$

这已经是现代 MoE 系统非常重要的基本结构。

---

# 三十五、Expert Parallelism

还有一个更大的问题：

如果：

$$
N=1000
$$

总参数巨大，不可能所有 Expert 都放在一张设备上。

于是：

$$
Device_1:E_1,\dots,E_{100}
$$

$$
Device_2:E_{101},\dots,E_{200}
$$

……

如果某 token：

$$
x\rightarrow E_{537}
$$

就把：

$$
x
$$

发送到拥有 \(E_{537}\) 的设备。

计算：

$$
E_{537}(x)
$$

再把结果返回。

这就是后来非常重要的：

$$
\boxed{\text{Expert Parallelism}}
$$

同时也产生现代 MoE 的一个核心瓶颈：

$$
\boxed{\text{Communication}}
$$

---

# 三十六、论文做了什么实验？

论文主要把 Sparsely-Gated MoE 用在：

$$
\boxed{\text{Language Modeling}}
$$

以及：

$$
\boxed{\text{Machine Translation}}
$$

等任务。

它当时还不是今天的 Transformer MoE；论文的实验架构主要建立在当时的 recurrent/LSTM 系统上，并将 MoE 层插入模型中。

它证明的重点不是：

> MoE 发明了一个更好的 FFN 激活函数。

而是：

$$
\boxed{
\text{模型容量可以大幅增长，而每个输入的计算成本不需要同比增长。}
}
$$

论文构建的模型最高达到约 **137B 参数**，这是当时极大的规模。

---

# 三十七、为什么这篇论文是现代 MoE 的起点？

因为现代 MoE 最核心的几个元素，在这里基本都出现了：

$$
\boxed{\text{Router}}
$$

$$
\boxed{\text{Top-K}}
$$

$$
\boxed{\text{Sparse Experts}}
$$

$$
\boxed{\text{Conditional Computation}}
$$

$$
\boxed{\text{Load Balancing}}
$$

$$
\boxed{\text{Expert Parallelism}}
$$

而最重要的 scaling 原则已经非常清楚：

$$
\boxed{
\text{Total Parameters}
\gg
\text{Activated Parameters}
}
$$

---

# 三十八、和 1991 MoE 做最终对比

### 1991

核心问题：

$$
\boxed{\text{如何让不同Expert自动分工？}}
$$

概率模型：

$$
p(y|x)
=
\sum_ip_iq_i
$$

Loss：

$$
L=-\log\sum_ip_iq_i
$$

责任：

$$
r_i
=
\frac{p_iq_i}
{\sum_jp_jq_j}
$$

核心思想：

$$
\boxed{\text{Competition + Specialization}}
$$

---

### 2017

核心问题：

$$
\boxed{\text{如何利用Expert分工实现超大模型的条件计算？}}
$$

MoE Layer：

$$
y
=
\sum_{i\in TopK}
G_i(x)E_i(x)
$$

Router：

$$
H_i
=
(xW_g)_i
+
\epsilon_iSoftplus((xW_{noise})_i)
$$

训练：

$$
L
=
L_{task}
+
L_{importance}
+
L_{load}
$$

核心思想：

$$
\boxed{
\text{Sparse Routing}
+
\text{Conditional Computation}
+
\text{Load Balancing}
}
$$

---

# 三十九、整篇论文真正需要记住的 8 个公式

如果你以后面试/复习，只需要把下面八个公式串起来。

**① Noisy Logit**

$$
\boxed{
H_i
=
(xW_g)_i
+
\epsilon_iSoftplus((xW_{noise})_i)
}
$$

**② Sparse Gate**

$$
\boxed{
G(x)
=
Softmax(KeepTopK(H(x),K))
}
$$

**③ MoE Output**

$$
\boxed{
y
=
\sum_iG_i(x)E_i(x)
}
$$

由于 sparse：

$$
y=
\sum_{i\in TopK}G_iE_i
$$

**④ Importance**

$$
\boxed{
Importance_i(X)
=
\sum_{x\in X}G_i(x)
}
$$

**⑤ Importance Loss**

$$
\boxed{
L_{importance}
=
w_{importance}CV(Importance)^2
}
$$

**⑥ Expert 进入 Top-K 的概率**

$$
\boxed{
P(i\in TopK)
=
\Phi
\left(
\frac{\mu_i-T_i}{\sigma_i}
\right)
}
$$

这是理解性简化形式；实际 threshold 处理会根据当前 Top-K 状态选择对应排名边界。

**⑦ Smooth Load**

$$
\boxed{
Load_i(X)
=
\sum_{x\in X}P(i\in TopK|x)
}
$$

以及：

$$
\boxed{
L_{load}
=
w_{load}CV(Load)^2
}
$$

**⑧ Total Training Objective**

$$
\boxed{
L_{total}
=
L_{task}
+
L_{importance}
+
L_{load}
}
$$

把这八个式子真正理解以后，2017 Sparse MoE 的算法主体基本就吃透了。

---

最后把全文压缩成一条因果链：

$$
\boxed{
\text{想增加模型容量}
}
$$

$$
\Downarrow
$$

$$
\text{增加大量Experts}
$$

$$
\Downarrow
$$

$$
\text{全部计算太贵}
$$

$$
\Downarrow
$$

$$
\boxed{\text{Top-K Sparse Routing}}
$$

$$
\Downarrow
$$

$$
\text{只激活K个Experts}
$$

$$
\Downarrow
$$

$$
\boxed{
\text{Parameters}\uparrow\uparrow,\quad
\text{FLOPs增长较慢}
}
$$

但：

$$
\Downarrow
$$

$$
\text{Router容易集中到少数Experts}
$$

所以：

$$
\boxed{
L_{importance}+L_{load}
}
$$

而 hard load 不可导：

$$
\Downarrow
$$

引入：

$$
\boxed{\text{Noisy Top-K}}
$$

使：

$$
H_i\sim N(\mu_i,\sigma_i^2)
$$

从而：

$$
\mathbf1[i\in TopK]
$$

变成：

$$
\boxed{
P(i\in TopK)
}
$$

获得平滑 Load estimator。

最后为了真正跑大：

$$
\boxed{
Dispatch
\rightarrow
Expert\ Parallelism
\rightarrow
Combine
}
$$

这就是 **2017 Sparsely-Gated MoE 从数学到工程的完整主线**。下一篇 GShard 可以直接在这套框架上理解：它并不是重新发明这些数学，而是开始解决“**如何把这套东西真正放进 Transformer，并在超大规模分布式系统上稳定运行**”。


# Sparse MoE：Noisy Top-K Routing 与可微 Load Balancing

## 1. Sparse MoE 的基本路由

Sparse MoE 中包含 \(E\) 个专家，但对于每个 token，只选择其中 \(K\) 个专家进行计算。

对于输入 token：

$$
x\in\mathbb R^d
$$

Router 首先产生每个专家对应的 routing score：

$$
\mu=xW_g
$$

其中：

$$
\mu\in\mathbb R^E
$$

这里的 \(\mu_i\) 是 Expert \(i\) 的 **logit / routing score**，并不是“Expert 被选中的概率”。

最简单的 Sparse MoE 可以直接：

$$
S=TopK(\mu,K)
$$

得到离散的专家集合。

例如：

$$
\mu=[2.1,1.8,0.3,-0.5],\quad K=2
$$

则：

$$
TopK(\mu)=\{E_1,E_2\}
$$

因此真正的专家选择是一个 **Hard Top-K 离散操作**。

---

# 2. Noisy Top-K Routing

经典 Noisy Top-K Gating 会进一步为 routing score 添加高斯噪声。

Router 除了产生：

$$
\mu_i
$$

还产生对应的噪声尺度：

$$
\sigma_i
=
Softplus((xW_{noise})_i)
$$

然后采样：

$$
\epsilon_i\sim\mathcal N(0,1)
$$

得到 noisy routing score：

$$
\boxed{
H_i=\mu_i+\sigma_i\epsilon_i
}
$$

因此从概率分布角度：

$$
\boxed{
H_i\sim\mathcal N(\mu_i,\sigma_i^2)
}
$$

实际路由则根据一次采样得到的 \(H\)：

$$
TopK(H,K)
$$

选择真正参与计算的专家。

因此完整过程是：

$$
\boxed{
x
\rightarrow
(\mu,\sigma)
\rightarrow
H=\mu+\sigma\epsilon
\rightarrow
TopK(H)
\rightarrow
K个专家
}
$$

这里需要区分两个概念：

**实际 Forward 路由**是离散的：

$$
Expert_i\in TopK
\quad\text{or}\quad
Expert_i\notin TopK
$$

而 \(H_i\) 本身是一个连续随机变量：

$$
H_i\sim\mathcal N(\mu_i,\sigma_i^2)
$$

这为后面的可微 Load 计算提供了基础。

---

# 3. 为什么直接统计 Expert Load 不可导

真实的 Expert Load 可以写成：

$$
Load_i^{hard}
=
\sum_{b=1}^{B}
\mathbf 1[i\in TopK(H_b)]
$$

其中：

$$
\mathbf 1[\cdot]
=
\begin{cases}
1,& Expert_i被选中\\
0,& Expert_i未被选中
\end{cases}
$$

例如一个 batch 中真实统计可能为：

$$
Load^{hard}
=
[31,27,4,2]
$$

问题在于：

$$
TopK+\mathbf 1[\cdot]
$$

是离散操作。

Router score 稍微变化，只要没有跨过 Top-K 边界：

$$
31\rightarrow31
$$

因此梯度几乎处处为 0，无法直接利用这种 hard count 优化 Router。

---

# 4. 用“进入 Top-K 的概率”替代 Hard Load

核心方法是把：

$$
\mathbf1[i\in TopK]
$$

替换成：

$$
P(i\in TopK)
$$

假设对于 Expert \(i\)，进入 Top-K 所需要超过的临界值为：

$$
T
$$

那么：

$$
i\in TopK
$$

可以转化为：

$$
H_i>T
$$

由于：

$$
H_i\sim\mathcal N(\mu_i,\sigma_i^2)
$$

因此：

$$
P(i\in TopK)
=
P(H_i>T)
$$

这就把一个离散事件转化成了连续概率。

---

# 5. Top-K 概率的推导

已知：

$$
H_i=\mu_i+\sigma_i\epsilon
$$

其中：

$$
\epsilon\sim\mathcal N(0,1)
$$

那么：

$$
P(H_i>T)
=
P(\mu_i+\sigma_i\epsilon>T)
$$

整理：

$$
P\left(
\epsilon>
\frac{T-\mu_i}{\sigma_i}
\right)
$$

利用标准正态分布 CDF：

$$
\Phi(z)=P(\epsilon\le z)
$$

得到：

$$
P(H_i>T)
=
1-\Phi\left(
\frac{T-\mu_i}{\sigma_i}
\right)
$$

由于标准正态分布对称：

$$
1-\Phi(-z)=\Phi(z)
$$

因此：

$$
\boxed{
P(i\in TopK)
=
\Phi\left(
\frac{\mu_i-T}{\sigma_i}
\right)
}
$$

这就是 Noisy Top-K 中计算 Expert 被选中概率的核心公式。

---

# 6. Continuous / Expected Load

对于 batch 中每一个 token，都可以计算：

$$
p_{bi}
=
P(Expert_i\in TopK\mid x_b)
$$

如果 batch 中有 \(B\) 个 token、\(E\) 个专家，则：

$$
P\in\mathbb R^{B\times E}
$$

例如：

$$
P=
\begin{bmatrix}
0.8&0.7&0.3&0.2\\
0.1&0.9&0.8&0.2\\
0.6&0.2&0.3&0.9
\end{bmatrix}
$$

然后对 token 求和：

$$
\boxed{
Load_i
=
\sum_{b=1}^{B}p_{bi}
}
$$

这个 Load 不一定是整数，例如：

$$
Load=[25.3,24.7,26.1,23.9]
$$

它表示的是：

$$
\boxed{
Load_i=\mathbb E[\text{Expert }i\text{ 接收到的 token 数量}]
}
$$

即 **Expected Load**，而不是真实的 hard token count。

---

# 7. Load Balancing Loss

得到每个专家的 Expected Load：

$$
Load=[Load_1,\cdots,Load_E]
$$

之后，可以使用变异系数平方：

$$
\boxed{
L_{load}=CV^2(Load)
}
$$

其中：

$$
CV^2(Load)
=
\frac{Var(Load)}
{Mean(Load)^2+\epsilon}
$$

如果：

$$
Load=[25,25,25,25]
$$

则方差接近 0，loss 很小。

如果：

$$
Load=[80,15,4,1]
$$

则方差很大，loss 增大。

因此它推动 Router 更均衡地使用不同 Expert。

---

# 8. 为什么 Gaussian CDF 可以计算这个概率

标准正态分布 PDF 为：

$$
\phi(x)
=
\frac1{\sqrt{2\pi}}
e^{-x^2/2}
$$

CDF 定义为：

$$
\boxed{
\Phi(x)
=
\int_{-\infty}^{x}
\frac1{\sqrt{2\pi}}
e^{-t^2/2}dt
}
$$

所以：

$$
P(H_i>T)
$$

本质上确实是在计算正态分布 PDF 下的一块面积。

但实际训练过程中**不会进行大量的数值积分**。

原因是 Gaussian CDF 可以通过特殊函数 `erf` 表示。

---

# 9. Error Function：erf

定义：

$$
\boxed{
erf(x)
=
\frac2{\sqrt\pi}
\int_0^x e^{-t^2}dt
}
$$

之所以引入 erf，是因为：

$$
\int e^{-x^2}dx
$$

无法使用有限组合的常见初等函数表示。

因此数学上专门定义了：

$$
erf(x)
$$

来表示这一类积分。

需要注意：

> **不是 Gaussian PDF 用 erf 表示，而是 Gaussian PDF 的积分，即 CDF，可以用 erf 表示。**

---

# 10. Gaussian CDF 与 erf 的关系

从：

$$
\Phi(x)
=
\frac1{\sqrt{2\pi}}
\int_{-\infty}^{x}
e^{-t^2/2}dt
$$

利用正态分布的对称性：

$$
\Phi(x)
=
\frac12+
\frac1{\sqrt{2\pi}}
\int_0^x e^{-t^2/2}dt
$$

令：

$$
u=\frac{t}{\sqrt2}
$$

则：

$$
t=\sqrt2u
$$

$$
dt=\sqrt2du
$$

并且：

$$
e^{-t^2/2}=e^{-u^2}
$$

因此：

$$
\Phi(x)
=
\frac12+
\frac1{\sqrt\pi}
\int_0^{x/\sqrt2}
e^{-u^2}du
$$

根据 erf 定义：

$$
erf(z)
=
\frac2{\sqrt\pi}
\int_0^z e^{-u^2}du
$$

最终得到：

$$
\boxed{
\Phi(x)
=
\frac12
\left[
1+
erf\left(\frac{x}{\sqrt2}\right)
\right]
}
$$

所以程序中可以直接：

```python
z = (mu - threshold) / sigma
prob = 0.5 * (1.0 + torch.erf(z / math.sqrt(2.0)))
```

而不需要自己进行数值积分。

---

# 11. 为什么 erf / CDF 可以反向传播

根据微积分基本定理：

$$
erf(x)
=
\frac2{\sqrt\pi}
\int_0^x e^{-t^2}dt
$$

因此：

$$
\boxed{
\frac{d}{dx}erf(x)
=
\frac2{\sqrt\pi}e^{-x^2}
}
$$

同理：

$$
\boxed{
\frac{d}{dx}\Phi(x)
=
\phi(x)
}
$$

即：

$$
\frac{d}{dx}\Phi(x)
=
\frac1{\sqrt{2\pi}}e^{-x^2/2}
$$

因此对于：

$$
p_i
=
\Phi\left(
\frac{\mu_i-T}{\sigma_i}
\right)
$$

令：

$$
z_i=
\frac{\mu_i-T}{\sigma_i}
$$

有：

$$
\frac{\partial p_i}{\partial\mu_i}
=
\frac1{\sigma_i}\phi(z_i)
$$

以及：

$$
\frac{\partial p_i}{\partial\sigma_i}
=
-\frac{\mu_i-T}{\sigma_i^2}\phi(z_i)
$$

所以梯度可以直接回传到 Router。

---

# 12. 为什么这里不存在 BPTT

虽然 CDF 的数学定义是一个积分：

$$
\Phi(x)=\int_{-\infty}^x\phi(t)dt
$$

但实际计算不是：

```text
积分点1 → 积分点2 → ... → 积分点N
```

而是直接调用：

```text
μ, σ
  ↓
z = (μ-T)/σ
  ↓
erf(z/√2)
  ↓
probability
```

因此计算图只是：

$$
(\mu,\sigma)
\rightarrow z
\rightarrow erf
\rightarrow p
$$

`erf` 在底层通过高效数值近似实现，而 Autograd 使用其已知解析导数：

$$
\frac{d}{dx}erf(x)
=
\frac2{\sqrt\pi}e^{-x^2}
$$

因此不存在时间展开，也不存在类似 RNN 的 BPTT。

---

# 13. Hard Routing 和 Soft Load 同时存在

这是整个设计最重要的结构。

**真正的模型 Forward：**

$$
H=\mu+\sigma\epsilon
$$

$$
TopK(H)
$$

得到：

$$
\boxed{\text{Hard Expert Selection}}
$$

只有选中的 K 个 Expert 真正计算，从而保持 Sparse MoE 的计算效率。

**Load Balancing：**

利用同一个随机变量的分布：

$$
H_i\sim N(\mu_i,\sigma_i^2)
$$

计算：

$$
P_i
=
P(H_i>T)
=
\Phi\left(
\frac{\mu_i-T}{\sigma_i}
\right)
$$

再得到：

$$
\boxed{
Load_i=\sum_bP_{bi}
}
$$

这是连续可导的 Expected Load。

因此：

$$
\boxed{
\begin{array}{ccc}
& H=\mu+\sigma\epsilon &\\
&\swarrow&\searrow\\
TopK(H)&&P(H_i>T)\\
\downarrow&&\downarrow\\
Hard\ Routing&&Soft/Expected\ Load\\
\downarrow&&\downarrow\\
Sparse\ Expert\ Compute&&Load\ Balancing\ Loss
\end{array}
}
$$

---

## 14. 整体梯度链

最终整个辅助 loss 的梯度链可以概括为：

$$
L_{load}
\rightarrow
Load_i
\rightarrow
P(i\in TopK)
\rightarrow
\Phi(z_i)
\rightarrow
z_i
\rightarrow
(\mu_i,\sigma_i)
\rightarrow
W_g,W_{noise}
$$

即：

$$
\boxed{
L_{load}
\rightarrow
\sum_bP_{bi}
\rightarrow
\Phi\left(\frac{\mu_i-T}{\sigma_i}\right)
\rightarrow
Router
}
$$

核心思想可以浓缩为：

$$
\boxed{
\text{Gaussian Noise}
+
\text{Hard Top-K}
\Rightarrow
\begin{cases}
\text{Forward：离散 Sparse Routing}\\
\text{Training：连续 Expected Load}
\end{cases}
}
$$

**Gaussian noise 一方面使专家选择具有随机性和探索性，另一方面给“是否进入 Top-K”建立了连续概率模型；再利用 Gaussian CDF / erf，就可以把不可导的 hard token count 转换成可导的 expected load，从而训练 Router 实现专家负载均衡。**